In [1]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"

In [2]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.metadata as package_metadata
import hashlib
import json
import re
import tempfile
import zipfile

import torch
import faiss
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

In [4]:
MODEL_ID = "google/embeddinggemma-2"
BATCH_SIZE = 4
EXPECTED_DOCUMENT_COUNT = 10  # 換自己的資料的大小
EXPORT_ZIP_NAME = "vector_ESG.zip"

# text 本身已含標題；這裡不再從 metadata 添加標題。
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.bfloat16
    if DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float32
)
PACKAGE_NAMES = ["torch", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
if DEVICE == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))

裝置： cpu ／模型精度： torch.float32


,套件,版本
0,torch,2.11.0+cpu
1,numpy,2.1.3
2,sentence-transformers,6.1.0
3,transformers,5.20.1
4,faiss-cpu,1.15.1
5,huggingface-hub,1.33.0


In [5]:
from pathlib import Path
import urllib.request
import io

archive_name = "ifrs_sds_2023_approved.zip"
zip_path = Path(archive_name)

# 如果本地找不到該 zip 檔（例如在 Google Colab 中執行），則自動從您的 GitHub 遠端下載
if not zip_path.is_file():
    print(f"本地找不到 {archive_name}，正在從 GitHub 專案自動下載...")
    # 請將下方網址替換為您 GitHub 上該 zip 檔的正確 Raw 網址
    github_raw_url = "https://raw.githubusercontent.com/Chuanglynne/RAG_esg/main/ifrs_sds_2023_approved.zip"
    urllib.request.urlretrieve(github_raw_url, archive_name)
    print("下載完成！")

# 直接讀取該 zip 檔的位元組資料，無縫對接後方的解壓縮與處理邏輯
archive_bytes = zip_path.read_bytes()
print(f"成功載入教材檔案：{archive_name}")

本地找不到 ifrs_sds_2023_approved.zip，正在從 GitHub 專案自動下載...
下載完成！
成功載入教材檔案：ifrs_sds_2023_approved.zip


In [7]:
import tempfile
import zipfile
from pathlib import Path

# 建立解壓縮目的地，並定義 DATA_DIR
DATA_DIR = Path(tempfile.mkdtemp(prefix="rag_data_"))
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR)

# 搜尋解壓縮後目錄下的所有 .txt 檔案
txt_paths = sorted(list(DATA_DIR.rglob("*.txt")))
if not txt_paths:
    raise ValueError("解壓後的目錄中找不到任何 .txt 檔案。")

# 計算解壓後的 SHA256 供後續 config.json 紀錄使用 (配合後面單元格的 SOURCE_ARCHIVE_SHA256)
import hashlib
SOURCE_ARCHIVE_SHA256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()

SAMPLE_INDEX = 0
sample_path = txt_paths[SAMPLE_INDEX]
try:
    sample_raw = sample_path.read_text(encoding="utf-8-sig")
except UnicodeDecodeError as exc:
    raise ValueError(f"{sample_path.name} 不是可讀取的 UTF-8 文字。") from exc
print("檔名：", sample_path.name)
print("=" * 50)
print(sample_raw)

檔名： 01_目的與範圍.txt.txt
標題：IFRS S1 目的與範圍（段次 1-9）
資料類型：官方資訊的教學改寫摘要（非網站全文、非法律意見）
主管機關：國際永續準則理事會 (ISSB) / 金融監督管理委員會
來源網址：https://ifrs.sfb.gov.tw/ifrs/home.cfm
教材整理日期：2026-11-01

【內容】
國際財務報導準則第S1號（IFRS S1）之目的，係規定個體揭露其對一般用途財務報告之主要使用者（現有及潛在投資者、貸款人及其他債權人）作成提供資源決策時有用之永續相關風險與機會資訊。個體應揭露可合理預期將於短期、中期或長期影響其現金流量、籌資可得性或資金成本之所有永續風險與機會。無論財務報表係依 IFRS 會計準則或其他 GAAP 編製，均可適用本準則。

【檢索練習問題】
IFRS S1 永續揭露準則的制定目的與適用範圍為何？

【提醒】
這是為 RAG 教學重寫的簡短摘要；完整適用條件、例外及最新修訂，請開啟來源網址核對。


In [9]:
def extract_rag_text(raw_text):
    """從本教材 TXT 取出標題與正文；換資料時修改這裡。"""
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    title_match = re.search(r"^標題[：:]\s*(.+)$", normalized, flags=re.MULTILINE)
    content_match = re.search(
        r"^【內容】[ \t]*\n(?P<body>.*?)(?=^【[^】\n]+】|\Z)",
        normalized,
        flags=re.MULTILINE | re.DOTALL,
    )
    if content_match is None or not content_match.group("body").strip():
        raise ValueError("找不到非空的【內容】正文；請修改 extract_rag_text() 以配合文件格式。")
    title = title_match.group(1).strip() if title_match else ""
    body = content_match.group("body").strip()
    return f"{title}\n\n{body}" if title else body

sample_rag_text = extract_rag_text(sample_raw)
print(sample_rag_text)
print("\n擷取後字元數：", len(sample_rag_text))

IFRS S1 目的與範圍（段次 1-9）

國際財務報導準則第S1號（IFRS S1）之目的，係規定個體揭露其對一般用途財務報告之主要使用者（現有及潛在投資者、貸款人及其他債權人）作成提供資源決策時有用之永續相關風險與機會資訊。個體應揭露可合理預期將於短期、中期或長期影響其現金流量、籌資可得性或資金成本之所有永續風險與機會。無論財務報表係依 IFRS 會計準則或其他 GAAP 編製，均可適用本準則。

擷取後字元數： 202


In [10]:
def extract_metadata(raw_text, filename):
    """取出本教材的描述資訊；換資料時修改這裡。"""
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    def read_field(name):
        match = re.search(rf"^{re.escape(name)}[：:][ \t]*(.*)$", normalized, flags=re.MULTILINE)
        return match.group(1).strip() or None if match else None

    return {
        "title": read_field("標題") or Path(filename).stem,
        "source_url": read_field("來源網址"),
        "date": read_field("教材整理日期"),
        "filename": filename,
        "document_type": read_field("資料類型") or "未標示",
    }

sample_metadata = extract_metadata(sample_raw, sample_path.relative_to(DATA_DIR).as_posix())
print(json.dumps(sample_metadata, ensure_ascii=False, indent=2))

{
  "title": "IFRS S1 目的與範圍（段次 1-9）",
  "source_url": "https://ifrs.sfb.gov.tw/ifrs/home.cfm",
  "date": "2026-11-01",
  "filename": "ifrs_sds_2023_approved/01_目的與範圍.txt.txt",
  "document_type": "官方資訊的教學改寫摘要（非網站全文、非法律意見）"
}


In [11]:
documents = []
for number, path in enumerate(txt_paths, start=1):
    filename = path.relative_to(DATA_DIR).as_posix()
    try:
        raw_text = path.read_text(encoding="utf-8-sig")
        if "\ufffd" in raw_text:
            raise ValueError("文字含有替代字元，請檢查原始編碼。")
        rag_text = extract_rag_text(raw_text)
        metadata = extract_metadata(raw_text, filename)
    except (ValueError, UnicodeDecodeError) as exc:
        raise ValueError(f"處理失敗：{filename}；原因：{exc}") from exc
    documents.append({"doc_id": f"doc_{number:03d}", "text": rag_text, "metadata": metadata})

if not documents:
    raise ValueError("沒有可建立索引的文件。")
index_doc_ids = [doc["doc_id"] for doc in documents]
if len(set(index_doc_ids)) != len(index_doc_ids):
    raise ValueError("doc_id 重複，請檢查編號規則。")

summary = pd.DataFrame([
    {"doc_id": doc["doc_id"], "標題": doc["metadata"]["title"],
     "正文含標題字元數": len(doc["text"]), "來源": doc["metadata"]["source_url"],
     "整理日期": doc["metadata"]["date"]}
    for doc in documents
])
display(summary)
for doc in documents:
    missing = [key for key in ("source_url", "date") if not doc["metadata"].get(key)]
    if missing:
        print("提醒：", doc["doc_id"], "缺少", ", ".join(missing))
print("第一筆統一格式：")
print(json.dumps(documents[0], ensure_ascii=False, indent=2))

,doc_id,標題,正文含標題字元數,來源,整理日期
0,doc_001,IFRS S1 目的與範圍（段次 1-9）,202,https://ifrs.sfb.gov.tw/ifrs/home.cfm,2026-11-01
1,doc_002,IFRS S1 觀念基礎、允當表達與重大性（段次 10-19）,199,https://ifrs.sfb.gov.tw/ifrs/home.cfm,2026-11-01
2,doc_003,IFRS S1 報導個體與連結之資訊（段次 20-24）,189,https://ifrs.sfb.gov.tw/ifrs/home.cfm,2026-11-01
3,doc_004,IFRS S1 核心內容 - 治理（段次 25-27）,169,https://ifrs.sfb.gov.tw/ifrs/home.cfm,2026-11-01
4,doc_005,IFRS S1 核心內容 - 策略與財務影響（段次 28-42）,227,https://ifrs.sfb.gov.tw/ifrs/home.cfm,2026-11-01
5,doc_006,IFRS S1 核心內容 - 風險管理（段次 43-44）,156,https://ifrs.sfb.gov.tw/ifrs/home.cfm,2026-11-01
6,doc_007,IFRS S1 核心內容 - 指標與目標（段次 45-53）,182,https://ifrs.sfb.gov.tw/ifrs/home.cfm,2026-11-01
7,doc_008,IFRS S1 一般規定 - 指引來源、位置與報導時點（段次 54-73）,213,https://ifrs.sfb.gov.tw/ifrs/home.cfm,2026-11-01
8,doc_009,IFRS S1 判斷、衡量不確定性與錯誤更正（段次 74-86）,156,https://ifrs.sfb.gov.tw/ifrs/home.cfm,2026-11-01
9,doc_010,IFRS S1 附錄 - 用語定義、品質特性與過渡規定（附錄 A-E）,190,https://ifrs.sfb.gov.tw/ifrs/home.cfm,2026-11-01


第一筆統一格式：
{
  "doc_id": "doc_001",
  "text": "IFRS S1 目的與範圍（段次 1-9）\n\n國際財務報導準則第S1號（IFRS S1）之目的，係規定個體揭露其對一般用途財務報告之主要使用者（現有及潛在投資者、貸款人及其他債權人）作成提供資源決策時有用之永續相關風險與機會資訊。個體應揭露可合理預期將於短期、中期或長期影響其現金流量、籌資可得性或資金成本之所有永續風險與機會。無論財務報表係依 IFRS 會計準則或其他 GAAP 編製，均可適用本準則。",
  "metadata": {
    "title": "IFRS S1 目的與範圍（段次 1-9）",
    "source_url": "https://ifrs.sfb.gov.tw/ifrs/home.cfm",
    "date": "2026-11-01",
    "filename": "ifrs_sds_2023_approved/01_目的與範圍.txt.txt",
    "document_type": "官方資訊的教學改寫摘要（非網站全文、非法律意見）"
  }
}


In [12]:
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID, token=False).sha
    model = SentenceTransformer(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        device=DEVICE,
        config_kwargs={"vision_config": None, "audio_config": None},
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
except Exception as exc:
    raise RuntimeError(
        "EmbeddingGemma 2 載入失敗。請確認網路、套件版本，必要時重新啟動工作階段；"
        "若 GPU 記憶體不足，改用 CPU。未替換成其他模型。"
    ) from exc

print("模型：", MODEL_ID)
print("版本：", MODEL_REVISION)
print("裝置：", model.device)
print("模型精度：", model.dtype)
print("向量維度：", model.get_sentence_embedding_dimension())
print("模型單筆 token 上限：", model.max_seq_length)

modules.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/22.2k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.49GB            

... (8 outputs left out)

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

模型： google/embeddinggemma-2
版本： f9972ac22e696a105c482202646979e33e4b5a1e
裝置： cpu
模型精度： torch.float32
向量維度： 768
模型單筆 token 上限： 1000000000000000019884624838656


/tmp/ipykernel_2726/465943176.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("向量維度：", model.get_sentence_embedding_dimension())


In [13]:
document_inputs = [DOCUMENT_TEMPLATE.format(text=doc["text"]) for doc in documents]
TOKEN_LIMIT = min(2048, int(model.max_seq_length))

if TOKEN_LIMIT <= 0:
    raise ValueError("輸入長度上限必須大於0。")

def count_input_tokens(text):
    return len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

token_counts = [count_input_tokens(text) for text in document_inputs]
length_table = pd.DataFrame({"doc_id": index_doc_ids, "tokens_含前綴": token_counts})
display(length_table)
long_docs = [doc_id for doc_id, length in zip(index_doc_ids, token_counts) if length > TOKEN_LIMIT]
if long_docs:
    raise ValueError(f"以下文件超過 {TOKEN_LIMIT} tokens，請先調整資料：{long_docs}")
print("本教材使用的 token 上限：", TOKEN_LIMIT)
print("全部文件均未超過上限；不需要截斷。")
print("第一筆真正送入 embedding 的文字：\n", document_inputs[0])

,doc_id,tokens_含前綴
0,doc_001,152
1,doc_002,160
2,doc_003,145
3,doc_004,130
4,doc_005,164
5,doc_006,117
6,doc_007,142
7,doc_008,157
8,doc_009,128
9,doc_010,157


本教材使用的 token 上限： 2048
全部文件均未超過上限；不需要截斷。
第一筆真正送入 embedding 的文字：
 title: none | text: IFRS S1 目的與範圍（段次 1-9）

國際財務報導準則第S1號（IFRS S1）之目的，係規定個體揭露其對一般用途財務報告之主要使用者（現有及潛在投資者、貸款人及其他債權人）作成提供資源決策時有用之永續相關風險與機會資訊。個體應揭露可合理預期將於短期、中期或長期影響其現金流量、籌資可得性或資金成本之所有永續風險與機會。無論財務報表係依 IFRS 會計準則或其他 GAAP 編製，均可適用本準則。


In [14]:
embeddings = model.encode(
    document_inputs,
    prompt="",
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError("向量形狀與文件數不符。")
if not np.isfinite(embeddings).all():
    raise ValueError("向量含有 NaN 或無限值。")
if np.any(np.linalg.norm(embeddings, axis=1) <= 0):
    raise ValueError("出現零向量。")
faiss.normalize_L2(embeddings)
DIMENSION = int(embeddings.shape[1])
if DIMENSION != int(model.get_sentence_embedding_dimension()):
    raise ValueError("實際向量維度與模型宣告不符。")
print("矩陣形狀（文件數, 向量維度）：", embeddings.shape)
print("第一篇向量的前10個數字：", embeddings[0, :10])
print("前5篇向量長度：", np.linalg.norm(embeddings[:5], axis=1))

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

矩陣形狀（文件數, 向量維度）： (10, 768)
第一篇向量的前10個數字： [ 0.06739339  0.01414832 -0.00328709 -0.00691604  0.06808561  0.03127215
 -0.02616447 -0.04557494  0.04724356 -0.14526364]
前5篇向量長度： [0.99999994 1.         1.         1.         1.        ]


/tmp/ipykernel_2726/2199939656.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  if DIMENSION != int(model.get_sentence_embedding_dimension()):


In [15]:
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)
documents_by_id = {doc["doc_id"]: doc for doc in documents}
assert index.ntotal == len(index_doc_ids) == len(documents)
print("索引類型：", type(index).__name__)
print("索引向量數：", index.ntotal)
print("索引維度：", index.d)
display(pd.DataFrame({"FAISS編號": range(len(index_doc_ids)), "doc_id": index_doc_ids}))

索引類型： IndexFlatIP
索引向量數： 10
索引維度： 768


,FAISS編號,doc_id
0,0,doc_001
1,1,doc_002
2,2,doc_003
3,3,doc_004
4,4,doc_005
5,5,doc_006
6,6,doc_007
7,7,doc_008
8,8,doc_009
9,9,doc_010


In [16]:
def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題超過模型輸入長度上限，請縮短問題。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    if np.linalg.norm(query_vector) <= 0:
        raise ValueError("問題產生零向量。")
    faiss.normalize_L2(query_vector)
    return query_vector


def search_documents(question, top_k=3):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於0的整數。")
    query_vector = embed_question(question)
    scores, positions = index.search(query_vector, min(int(top_k), index.ntotal))
    results = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        doc_id = index_doc_ids[int(position)]
        results.append({"rank": rank, "score": float(score), **documents_by_id[doc_id]})
    return results

In [17]:
QUESTION = "IFRS S1 永續揭露準則的制定目的與適用範圍為何？"
TOP_K = 3
results = search_documents(QUESTION, TOP_K)
display(pd.DataFrame([
    {"排名": r["rank"], "doc_id": r["doc_id"], "標題": r["metadata"]["title"],
     "cosine_similarity": r["score"], "來源": r["metadata"]["source_url"]}
    for r in results
]))
for result in results:
    print(f"\n[{result['doc_id']}] {result['metadata']['title']} ({result['score']:.4f})")
    print(result["text"])
    print("來源：", result["metadata"]["source_url"] or "未提供")

,排名,doc_id,標題,cosine_similarity,來源
0,1,doc_008,IFRS S1 一般規定 - 指引來源、位置與報導時點（段次 54-73）,0.866825,https://ifrs.sfb.gov.tw/ifrs/home.cfm
1,2,doc_001,IFRS S1 目的與範圍（段次 1-9）,0.852441,https://ifrs.sfb.gov.tw/ifrs/home.cfm
2,3,doc_007,IFRS S1 核心內容 - 指標與目標（段次 45-53）,0.847774,https://ifrs.sfb.gov.tw/ifrs/home.cfm



[doc_008] IFRS S1 一般規定 - 指引來源、位置與報導時點（段次 54-73） (0.8668)
IFRS S1 一般規定 - 指引來源、位置與報導時點（段次 54-73）

當缺乏明確可適用之 IFRS 永續準則時，個體應參考 SASB 準則中之揭露主題與指標，並可考量 CDSB 架構應用指引、GRI 或 ESRS。永續揭露應作為一般用途財務報告之一部分（如管理階層評論），報導時點與涵蓋期間應與財務報表相同（通常為 12 個月）。符合所有規定者應作明確無保留之遵循聲明。法令禁止或涉及商業敏感之極端情況可享有限度豁免。
來源： https://ifrs.sfb.gov.tw/ifrs/home.cfm

[doc_001] IFRS S1 目的與範圍（段次 1-9） (0.8524)
IFRS S1 目的與範圍（段次 1-9）

國際財務報導準則第S1號（IFRS S1）之目的，係規定個體揭露其對一般用途財務報告之主要使用者（現有及潛在投資者、貸款人及其他債權人）作成提供資源決策時有用之永續相關風險與機會資訊。個體應揭露可合理預期將於短期、中期或長期影響其現金流量、籌資可得性或資金成本之所有永續風險與機會。無論財務報表係依 IFRS 會計準則或其他 GAAP 編製，均可適用本準則。
來源： https://ifrs.sfb.gov.tw/ifrs/home.cfm

[doc_007] IFRS S1 核心內容 - 指標與目標（段次 45-53） (0.8478)
IFRS S1 核心內容 - 指標與目標（段次 45-53）

指標與目標面揭露之目的，係使使用者了解個體與永續風險與機會有關之績效及目標進展。個體應揭露適用之 IFRS 永續準則規定之指標、個體自訂或來自外部來源之指標（須註明定義、計算方法、是否經第三方驗證）。對每一設定或法令規範之目標，應揭露用以監控之指標、目標數值、適用期間、基準期、里程碑及績效趨勢分析。
來源： https://ifrs.sfb.gov.tw/ifrs/home.cfm


In [18]:
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="rag_index_"))
faiss.write_index(index, str(EXPORT_DIR / "index.faiss"))

for filename, data in [("documents.json", documents), ("index_doc_ids.json", index_doc_ids)]:
    (EXPORT_DIR / filename).write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )

config = {
    "format_version": "1.0",
    "created_at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds"),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_config_kwargs": {"vision_config": None, "audio_config": None},
    "dimension": DIMENSION,
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "metric": "cosine_similarity_via_normalized_inner_product",
    "document_template": DOCUMENT_TEMPLATE,
    "query_template": QUERY_TEMPLATE,
    "encode_prompt": "",
    "max_seq_length": TOKEN_LIMIT,
    "document_count": len(documents),
    "document_unit": "one_document_one_vector",
    "document_text_includes_title": True,
    "package_versions": PACKAGE_VERSIONS,
    "build_device": DEVICE,
    "build_dtype": str(DTYPE),
    "source_archive": archive_name,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "file_sha256": {
        name: hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
        for name in ["index.faiss", "documents.json", "index_doc_ids.json"]
    },
}
(EXPORT_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("已保存：")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")
print("\n模型版本及前綴已寫入 config.json。")

已保存：
config.json (1,386 bytes)
documents.json (9,288 bytes)
index.faiss (30,765 bytes)
index_doc_ids.json (132 bytes)

模型版本及前綴已寫入 config.json。


In [19]:
loaded_config = json.loads((EXPORT_DIR / "config.json").read_text(encoding="utf-8"))
for name, expected_hash in loaded_config["file_sha256"].items():
    actual_hash = hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"檔案完整性檢查失敗：{name}"

loaded_index = faiss.read_index(str(EXPORT_DIR / "index.faiss"))
loaded_documents = json.loads((EXPORT_DIR / "documents.json").read_text(encoding="utf-8"))
loaded_ids = json.loads((EXPORT_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
assert loaded_index.d == loaded_config["dimension"]
assert len(set(loaded_ids)) == len(loaded_ids)
assert set(loaded_ids) == {doc["doc_id"] for doc in loaded_documents}
assert loaded_documents == documents and loaded_ids == index_doc_ids

verification_query = embed_question(QUESTION)
k = min(TOP_K, index.ntotal)
before_scores, before_positions = index.search(verification_query, k)
after_scores, after_positions = loaded_index.search(verification_query, k)
np.testing.assert_array_equal(before_positions, after_positions)
np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)

all_scores = (embeddings @ verification_query[0]).ravel()
expected_top_scores = np.sort(all_scores)[::-1][:k]
np.testing.assert_allclose(after_scores[0], expected_top_scores, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(after_scores[0], all_scores[after_positions[0]], atol=1e-5, rtol=1e-5)
print("通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。")

通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。


In [21]:
from google.colab import files

EXPORT_ZIP = EXPORT_DIR.parent / f"{EXPORT_DIR.name}_download" / EXPORT_ZIP_NAME
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)
required_files = ["index.faiss", "documents.json", "index_doc_ids.json", "config.json"]
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(EXPORT_DIR / name, arcname=name)
with zipfile.ZipFile(EXPORT_ZIP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)
print("下載檔名：", EXPORT_ZIP_NAME)
print("大小：", f"{EXPORT_ZIP.stat().st_size:,} bytes")
files.download(str(EXPORT_ZIP))

下載檔名： vector_ESG.zip
大小： 32,634 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>